In [ ]:
from statsbombpy import sb
import pandas as pd

matches = sb.matches(competition_id=11, season_id=27)
match_ids = matches['match_id'].tolist()

all_events = []
for match_id in match_ids:
    events = sb.events(match_id=match_id)
    all_events.append(events)

events_df = pd.concat(all_events, ignore_index=True)
print(f"Total events loaded: {len(events_df)}")

In [ ]:
import os
save_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data', 'la_liga_1516_events.parquet')
events_df.to_parquet(save_path, index=False)
print(f"Saved to: {save_path}")
print(f"File exists: {os.path.exists(save_path)}")

In [ ]:
# StatsBomb coordinates are in yards. Pitch dimensions: 120 x 80 yards.
# Goal center: [120, 40]. All distance calculations produce values in yards.

shots = events_df[events_df['type']  == 'Shot'].copy()
print(f"Total shots: {len(shots)}")
print(shots[['player', 'location', 'shot_statsbomb_xg']].head(10))

Total shots: 9168
                            player       location  shot_statsbomb_xg
2356  Jefferson Andrés Lerma Solís  [112.5, 42.1]           0.100429
2357        Sergio Gontán Gallardo   [90.9, 39.9]           0.023138
2358       Adrián González Morales  [103.1, 52.5]           0.034535
2359          Borja González Tomás  [116.6, 38.6]           0.394774
2360                  Nabil Ghilas  [112.4, 35.8]           0.077194
2361                  Nabil Ghilas  [102.6, 39.7]           0.072501
2362                  Takashi Inui  [102.6, 30.4]           0.049790
2363       Adrián González Morales  [109.9, 32.8]           0.151259
2364                  Nabil Ghilas   [99.1, 38.2]           0.074698
2365   José Antonio García Rabasco  [106.1, 28.7]           0.045668


In [10]:
shots['x'] = shots['location'].apply(lambda loc: loc[0])
shots['y'] = shots['location'].apply(lambda loc: loc[1])

GOAL_X = 120
GOAL_Y = 40

shots['shot_distance'] = ((shots['x'] - GOAL_X)**2 + (shots['y'] - GOAL_Y)**2)**0.5

print(shots['shot_distance'].describe())

count    9168.000000
mean       19.046284
std         8.685988
min         0.632456
25%        12.000417
50%        18.261709
75%        25.269300
max        74.867416
Name: shot_distance, dtype: float64


In [6]:
mean_dist    = shots['shot_distance'].mean()
median_dist  = shots['shot_distance'].median()
std_dist     = shots['shot_distance'].std()

print(f"Mean shot distance:   {mean_dist:.2f}")
print(f"Median shot distance: {median_dist:.2f}")
print(f"Std deviation:        {std_dist:.2f}")

Mean shot distance:   19.05
Median shot distance: 18.26
Std deviation:        8.69


In [7]:
shots['distance_percentile'] = shots['shot_distance'].rank(pct=True) * 100

for p in [25, 50, 75, 90]:
    val = shots['shot_distance'].quantile(p/100)
    print(f"{p}th percentile: {val:.2f}")

25th percentile: 12.00
50th percentile: 18.26
75th percentile: 25.27
90th percentile: 30.04


In [ ]:
from statsbombpy import sb
import pandas as pd

matches = sb.matches(competition_id=11, season_id=27)
all_match_ids = matches['match_id'].tolist()
matches_df = pd.DataFrame(matches)
print(f'Total matches in La Liga 2015/16: {len(all_match_ids)}')

In [ ]:
import os
save_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data', 'la_liga_1516_matches.parquet')
matches_df.to_parquet(save_path, index=False)
print(f"Saved to: {save_path}")
print(f"File exists: {os.path.exists(save_path)}")

In [ ]:
from statsbombpy import sb

# Use any known Real Madrid match_id from your matches parquet
# The example match object you shared has match_id 3825833
lineups = sb.lineups(match_id=3825833)

print(type(lineups))              # should be dict
print(lineups.keys())             # should show both team names

# Inspect one team
rm_lineup = lineups['Real Madrid']
print(type(rm_lineup))            # should be DataFrame
print(rm_lineup.columns.tolist()) # THIS is what you need — paste the output here
print(rm_lineup.head(2))

print(rm_lineup['positions'].iloc[0])  # should be a list of dicts like the example you shared
print(type(rm_lineup['positions'].iloc[0]))

print(rm_lineup[['player_name', 'player_nickname']].to_string())

In [ ]:
from statsbombpy import sb
import pandas as pd
import os

matches = pd.read_parquet('data/la_liga_1516_matches.parquet')
all_match_ids = matches['match_id'].tolist()

lineup_records = []

for match_id in all_match_ids:
    lineups = sb.lineups(match_id=match_id)
    # lineups is a dict: {team_name: DataFrame}
    for team_name, team_df in lineups.items():
        team_df = team_df.copy()
        team_df['match_id'] = match_id
        team_df['team'] = team_name
        lineup_records.append(team_df)

lineups_df = pd.concat(lineup_records, ignore_index=True)

save_path = os.path.join('data', 'la_liga_1516_lineups.parquet')
os.makedirs('data', exist_ok=True)
lineups_df.to_parquet(save_path, index=False)

print(f'Saved: {save_path}')
print(f'Shape: {lineups_df.shape}')
print(f'Columns: {lineups_df.columns.tolist()}')

In [9]:
lineups_df = pd.read_parquet('data/la_liga_1516_lineups.parquet')

# Find Ronaldo across all matches
ronaldo = lineups_df[lineups_df['player_name'] == 'Cristiano Ronaldo dos Santos Aveiro']
print(f'Ronaldo appears in {len(ronaldo)} matches')

# Inspect positions for one match
sample_positions = ronaldo['positions'].iloc[0]
print(sample_positions)

# Test the minutes extraction logic on this one entry
def parse_match_minute(time_str, period):
    """Convert a within-period timestamp and period number to a match minute."""
    if time_str is None:
        # None means Final Whistle — assume end of period 2
        return 90.0
    mins, secs = time_str.split(':')
    within_period = int(mins) + int(secs) / 60
    period_offset = (int(period) - 1) * 45
    return period_offset + within_period

def extract_minutes(positions_list):
    """
    Extract minutes played from a StatsBomb positions list.
    Handles within-period timestamps by converting to match minutes using period number.
    Returns minutes as a float.
    """
    # Guard against numpy arrays and empty values from parquet serialization
    if positions_list is None:
        return 0.0
    if not isinstance(positions_list, list):
        try:
            positions_list = list(positions_list)
        except Exception:
            return 0.0
    if len(positions_list) == 0:
        return 0.0

    first = positions_list[0]
    last = positions_list[-1]

    entry_minute = parse_match_minute(first.get('from'), first.get('from_period', 1))

    # Exit: use last position's 'to' and 'to_period'
    to_time = last.get('to')
    to_period = last.get('to_period')

    if to_time is None or to_period is None:
        exit_minute = 90.0
    else:
        exit_minute = parse_match_minute(to_time, to_period)

    return round(exit_minute - entry_minute, 2)

ronaldo_minutes = ronaldo['positions'].apply(extract_minutes).sum()
print(f'Ronaldo total minutes: {ronaldo_minutes:.1f}')
print(f'Ronaldo season 90s: {ronaldo_minutes / 90:.2f}')

Ronaldo appears in 36 matches
[{'end_reason': 'Substitution - Off (Tactical)', 'from': '00:00', 'from_period': 1, 'position': 'Left Wing', 'position_id': 21, 'start_reason': 'Starting XI', 'to': '45:00', 'to_period': 2.0}]
Ronaldo total minutes: 3273.7
Ronaldo season 90s: 36.37
